# ATP Match Prediction — Inference Pipeline

Saved production modellerini yükleyip tek maç için
win probability üretme.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

In [2]:
project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

models_directory = project_root / "models"

processed_data_directory = (
    project_root / "data" / "processed"
)


xgboost_model = joblib.load(
    models_directory
    / "xgboost_pipeline.joblib"
)

logistic_model = joblib.load(
    models_directory
    / "logistic_pipeline.joblib"
)

ensemble_config = json.loads(
    (
        models_directory
        / "ensemble_config.json"
    ).read_text(encoding="utf-8")
)

feature_registry = json.loads(
    (
        processed_data_directory
        / "feature_registry.json"
    ).read_text(encoding="utf-8")
)

model_data = pd.read_pickle(
    processed_data_directory
    / "atp_model_data_1990_2026.pkl"
)


model_feature_columns = feature_registry[
    "model_feature_columns"
]


print("Models loaded")
print("Required features:", len(model_feature_columns))
print("Available matches:", len(model_data))

Models loaded
Required features: 119
Available matches: 95898


In [3]:
def get_confidence_label(confidence):
    if confidence < 0.60:
        return "Toss-up"

    if confidence < 0.70:
        return "Lean"

    if confidence < 0.80:
        return "Strong pick"

    return "Heavy favorite"


def predict_match_frame(match_frame):
    if len(match_frame) != 1:
        raise ValueError(
            "match_frame must contain exactly one match"
        )

    missing_features = set(
        model_feature_columns
    ) - set(match_frame.columns)

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    X_match = match_frame[
        model_feature_columns
    ]

    xgboost_probability = (
        xgboost_model.predict_proba(
            X_match
        )[0, 1]
    )

    logistic_probability = (
        logistic_model.predict_proba(
            X_match
        )[0, 1]
    )

    p1_probability = (
        ensemble_config["xgboost_weight"]
        * xgboost_probability
        + ensemble_config["logistic_weight"]
        * logistic_probability
    )

    p2_probability = 1 - p1_probability

    match = match_frame.iloc[0]

    if p1_probability >= p2_probability:
        predicted_winner = match["p1_name"]
        confidence = p1_probability
    else:
        predicted_winner = match["p2_name"]
        confidence = p2_probability

    return {
        "p1_name": match["p1_name"],
        "p2_name": match["p2_name"],
        "p1_win_probability": round(
            float(p1_probability),
            4,
        ),
        "p2_win_probability": round(
            float(p2_probability),
            4,
        ),
        "predicted_winner": predicted_winner,
        "confidence": round(
            float(confidence),
            4,
        ),
        "confidence_label": (
            get_confidence_label(confidence)
        ),
    }

In [4]:
example_match = model_data.tail(1).copy()

prediction = predict_match_frame(
    example_match
)

prediction

{'p1_name': 'Flavio Cobolli',
 'p2_name': 'Alexander Zverev',
 'p1_win_probability': 0.2218,
 'p2_win_probability': 0.7782,
 'predicted_winner': 'Alexander Zverev',
 'confidence': 0.7782,
 'confidence_label': 'Strong pick'}